# Coastal flood step 03: post-processing and summaries

Builds grouped asset damage layers and sector/subsector/return-period summary tables.


In [ ]:
import os
from glob import glob
import geopandas
import pandas
import subprocess
from pathlib import Path
import sys
import numpy
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.ticker import FuncFormatter

# !{sys.executable} -m pip install "nismod-snail==0.5.3"

root = Path("/Users/robynhaggis/Documents/Geospatial_analysis/dphil_papers")
if str(root) not in sys.path:
    sys.path.insert(0, str(root))



In [ ]:
# processed_data_path = 'L:\Jamaica\Inputs'
base_path = Path("/Users/robynhaggis/Documents/Geospatial_analysis/dphil_papers")
output_path = base_path / "dphil_paper_3/results"
processed_data_path = base_path / "dphil_paper_3/processed_data"
networks_path = base_path / "dphil_common_cross_cutting/common_incoming_data/networks/networks"
damage_curves_path = base_path / "dphil_common_cross_cutting/common_incoming_data/damage_curves"
robyn_libraries_path = base_path / "robyns_libraries"
data_root = base_path / "dphil_common_cross_cutting/common_incoming_data"

jamaica_crs = 3448
jamaica_metric_grid_crs = "EPSG:3448"

mangroves_shapefile_candidates = [
    base_path / "dphil_paper_3/inputs/forces_of_nature_mangroves/mangroves.shp",
    data_root / "landcover/mangroves_fn/mangroves.shp",
]
mangroves_shapefile = next((path for path in mangroves_shapefile_candidates if path.exists()), None)
if mangroves_shapefile is None:
    raise FileNotFoundError(
        "Could not find mangroves shapefile. Checked:\n" + "\n".join(str(path) for path in mangroves_shapefile_candidates)
    )

def resolve_network_asset_file(asset_relative_path):
    relative_asset_path = Path(asset_relative_path)
    asset_file_in_common_incoming_data = data_root / relative_asset_path
    asset_file_in_nested_networks_folder = data_root / "networks" / relative_asset_path

    if asset_file_in_common_incoming_data.exists():
        return asset_file_in_common_incoming_data
    if asset_file_in_nested_networks_folder.exists():
        return asset_file_in_nested_networks_folder

    raise FileNotFoundError(
        f"Could not find asset file '{relative_asset_path}'. Checked: \n"
        f"- {asset_file_in_common_incoming_data}\n"
        f"- {asset_file_in_nested_networks_folder}"
    )

In [ ]:
network_csv = data_root / "networks/network_layers_hazard_intersections_details.csv" 
hazard_csv = base_path / "dphil_paper_3/inputs/coastal_flood_rasters.csv"
damage_curves_csv = damage_curves_path / "asset_damage_curve_mapping.csv"
hazard_damage_parameters_csv = damage_curves_path / "hazard_damage_parameters.csv"
damage_results_folder = base_path / "dphil_paper_3/processed_data/direct_damages"

In [ ]:
damage_results_folder = os.path.join(output_path, "direct_damages")
mangrove_flood_damage_columns = ["coastal_flood_fn_mg_rp_25",
                       "coastal_flood_fn_mg_rp_100",
                       "coastal_flood_fn_mg_rp_500"]
nomangrove_flood_damage_columns = ["coastal_flood_fn_nomg_rp_25",
                       "coastal_flood_fn_nomg_rp_100",
                       "coastal_flood_fn_nomg_rp_500"]
difference_columns = ["coastal_flood_diff_rp_25",
                       "coastal_flood_diff_rp_100",
                       "coastal_flood_diff_rp_500"]
flood_damage_columns = mangrove_flood_damage_columns + nomangrove_flood_damage_columns + difference_columns 


In [ ]:
asset_data_details = pandas.read_csv(network_csv)
damage_totals = [] # List object to assemble many dataframes 
damage_estimates_directory = Path(output_path) / "damage_estimates"
damage_estimates_directory.mkdir(parents=True, exist_ok=True)
for asset_info in asset_data_details.itertuples():
        asset_path = asset_info.path
        asset_gpkg = asset_info.asset_gpkg
        asset_layer = asset_info.asset_layer
        asset_id = asset_info.asset_id_column
        df_path = os.path.join(output_path,"direct_damages",
                               f"{asset_gpkg}_{asset_layer}",
                               f"{asset_gpkg}_{asset_layer}_direct_damages_parameter_set_0.parquet")
        if os.path.exists(df_path):
            df = pandas.read_parquet(df_path) #read in the files using the .parquet from Raghav's code
            df[difference_columns] = df[nomangrove_flood_damage_columns] - df[mangrove_flood_damage_columns].values
            df = df.groupby([asset_id]).sum(flood_damage_columns).reset_index() # calculate asset level damages = .groupby(node_id).sum()
            asset_relative_path = asset_info.path
            resolved_asset_file = resolve_network_asset_file(asset_relative_path)
            df_geom = geopandas.read_file(resolved_asset_file, layer=asset_layer)
            df_geom = df_geom.to_crs(epsg=jamaica_crs) # Convert geometry to Jamaica projection system
            df = pandas.merge(df,df_geom[[asset_id,"geometry"]],how="left",on=[asset_id])
            df = geopandas.GeoDataFrame(df,geometry="geometry",crs=jamaica_crs)
            print(df)
            output_geopackage = damage_estimates_directory / f"{asset_gpkg}_{asset_layer}_asset_damages_groupedby.gpkg"
            df.to_file(output_geopackage, driver="GPKG")
            df['sector'] = asset_gpkg
            df['layer'] = asset_layer
            df = df.groupby(['sector','layer']).sum(flood_damage_columns).reset_index()
            damage_totals.append(df) # Add things to list

# Convert list of dataframes to 1 dataframe by concatenation
damage_totals = pandas.concat(damage_totals,axis=0,ignore_index=True)
damage_totals.to_csv(damage_estimates_directory / "asset_damages_groupedby.csv")


## Post-processing (clean and reproducible)
The cells below replace old duplicate/debug cells and give reproducible outputs.

In [ ]:
damage_estimates_directory = Path(output_path) / "damage_estimates"
if not damage_estimates_directory.exists():
    raise FileNotFoundError(f"Missing folder: {damage_estimates_directory}")

damage_estimate_files = sorted(damage_estimates_directory.glob("*_asset_damages_groupedby.gpkg"))
if not damage_estimate_files:
    raise FileNotFoundError(f"No grouped damage GPKGs found in {damage_estimates_directory}")

summary_rows = []
for grouped_damage_file in damage_estimate_files:
    grouped_damage = geopandas.read_file(grouped_damage_file)
    asset_name = grouped_damage_file.stem.replace("_asset_damages_groupedby", "")
    summary_rows.append({
        "asset_name": asset_name,
        "row_count": len(grouped_damage),
        "column_count": len(grouped_damage.columns),
    })

pandas.DataFrame(summary_rows).sort_values("asset_name").reset_index(drop=True)


In [ ]:
# Choose one output to inspect (this reproduces table outputs like the old debug cells).
asset_name_to_preview = "pipelines_NWC_edges"  # e.g. rail_nodes, roads_edges, buildings_assigned_economic_activity_areas
preview_file = damage_estimates_directory / f"{asset_name_to_preview}_asset_damages_groupedby.gpkg"

if not preview_file.exists():
    raise FileNotFoundError(f"Missing file: {preview_file}")

preview_table = geopandas.read_file(preview_file)
print(f"Rows: {len(preview_table)} | Columns: {len(preview_table.columns)}")
preview_table.head(20)


In [ ]:
# Reproducible mangrove-to-asset mapping for one asset layer.
asset_gpkg_for_mapping = "roads"
asset_layer_for_mapping = "edges"
asset_data_details = pandas.read_csv(network_csv)
asset_row = asset_data_details.loc[
    (asset_data_details.asset_gpkg == asset_gpkg_for_mapping)
    & (asset_data_details.asset_layer == asset_layer_for_mapping)
].squeeze()

if asset_row.empty:
    raise ValueError("No matching asset row found in network CSV")

asset_id_column = asset_row.asset_id_column
asset_file = resolve_network_asset_file(asset_row.path)
asset_geometry = geopandas.read_file(asset_file, layer=asset_layer_for_mapping)[[asset_id_column, "geometry"]]
asset_geometry = asset_geometry.to_crs(epsg=jamaica_crs)

mangroves_geometry = geopandas.read_file(mangroves_shapefile)[["ID", "geometry"]]
mangroves_geometry = mangroves_geometry.to_crs(epsg=jamaica_crs)

mangrove_asset_mapping = geopandas.sjoin(
    mangroves_geometry,
    asset_geometry,
    how="inner",
    predicate="intersects",
)[["ID", asset_id_column]].drop_duplicates().reset_index(drop=True)

print(f"Mapped pairs: {len(mangrove_asset_mapping)}")
if len(mangrove_asset_mapping) == 0:
    print("No intersections found for this asset selection. Try roads/edges or buildings_assigned_economic_activity/areas.")
mangrove_asset_mapping.head(30)


In [ ]:
# Sum damages by Sector, Subsector, and ReturnPeriod.
# Avoided damages are signed: Without_Mangroves - With_Mangroves (can be negative).
network_details = pandas.read_csv(network_csv)[["sector", "asset_description", "asset_gpkg", "asset_layer"]].drop_duplicates()

damage_rows = []
for asset_info in network_details.itertuples(index=False):
    damage_file = (
        Path(output_path)
        / "direct_damages"
        / f"{asset_info.asset_gpkg}_{asset_info.asset_layer}"
        / f"{asset_info.asset_gpkg}_{asset_info.asset_layer}_direct_damages_parameter_set_0.parquet"
    )

    if not damage_file.exists():
        continue

    damage_table = pandas.read_parquet(damage_file)
    return_periods = sorted({col.split("_rp_")[-1] for col in damage_table.columns if "_rp_" in col})

    for return_period in return_periods:
        mg_col = f"coastal_flood_fn_mg_rp_{return_period}"
        nomg_col = f"coastal_flood_fn_nomg_rp_{return_period}"
        nomg_minus_mg_col = f"coastal_flood_fn_nomg_minus_mg_rp_{return_period}"

        damages_with_mangroves = float(damage_table[mg_col].sum()) if mg_col in damage_table.columns else 0.0

        if nomg_col in damage_table.columns:
            damages_without_mangroves = float(damage_table[nomg_col].sum())
        elif nomg_minus_mg_col in damage_table.columns:
            damages_without_mangroves = damages_with_mangroves + float(damage_table[nomg_minus_mg_col].sum())
        else:
            damages_without_mangroves = 0.0

        avoided_damages = damages_without_mangroves - damages_with_mangroves
        nomg_minus_mg_raster_damages = float(damage_table[nomg_minus_mg_col].sum()) if nomg_minus_mg_col in damage_table.columns else float('nan')

        damage_rows.append({
            "Sector": asset_info.sector,
            "Subsector": asset_info.asset_description,
            "Asset": asset_info.asset_gpkg,
            "Layer": asset_info.asset_layer,
            "ReturnPeriod": int(return_period),
            "Damages_With_Mangroves_JD": damages_with_mangroves,
            "Damages_Without_Mangroves_JD": damages_without_mangroves,
            "Avoided_Damages_JD": avoided_damages,
            "NomgMinusMg_Raster_Damages_JD": nomg_minus_mg_raster_damages,
        })

asset_level_summary = pandas.DataFrame(damage_rows)

sector_subsector_summary = (
    asset_level_summary
    .groupby(["Sector", "Subsector", "ReturnPeriod"], as_index=False)[
        ["Damages_With_Mangroves_JD", "Damages_Without_Mangroves_JD", "Avoided_Damages_JD", "NomgMinusMg_Raster_Damages_JD"]
    ]
    .sum()
    .sort_values(["Sector", "Subsector", "ReturnPeriod"])
)

sector_summary = (
    sector_subsector_summary
    .groupby(["Sector", "ReturnPeriod"], as_index=False)[
        ["Damages_With_Mangroves_JD", "Damages_Without_Mangroves_JD", "Avoided_Damages_JD", "NomgMinusMg_Raster_Damages_JD"]
    ]
    .sum()
    .sort_values(["Sector", "ReturnPeriod"])
)

sector_subsector_summary_file = Path(output_path) / "damage_estimates" / "sector_subsector_return_period_damages.csv"
sector_subsector_summary.to_csv(sector_subsector_summary_file, index=False)

print(f"Saved: {sector_subsector_summary_file}")
print("\nSector + Subsector summary:")
display(sector_subsector_summary)
print("\nSector-only summary:")
display(sector_summary)
